# Entropy-Based Analytic Model

This notebook generates a synthetic dataset and performs entropy-based analytical exploration.

**Dataset columns:**
- `text_message` – Short example text messages
- `url` – Example URLs
- `language` – Language of the message
- `country` – Country of origin
- `device` – Device type used
- `star_review` – 5-star review rating (1–5)

## 1. Imports

In [ ]:
import random
import statistics

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import entropy as scipy_entropy

%matplotlib inline
pd.set_option('display.max_columns', None)

## 2. Generate Synthetic Dataset (1000 rows × 6 columns)

In [ ]:
random.seed(42)
np.random.seed(42)

N = 1000

# --- Text messages ---
text_templates = [
    "Great product, really happy with it!",
    "Shipping was delayed but item is fine.",
    "Would not recommend to a friend.",
    "Exactly as described, fast delivery.",
    "Poor quality, broke after one week.",
    "Amazing experience, will buy again!",
    "Customer support was very helpful.",
    "The item looks different from the photo.",
    "Decent value for the price.",
    "Arrived damaged, requesting a refund.",
    "Exceeded my expectations!",
    "Takes too long to load on my device.",
    "Easy to use and intuitive interface.",
    "Instructions were hard to follow.",
    "Perfect gift for the holidays.",
    "Not worth the money.",
    "Works as advertised.",
    "Packaging was damaged on arrival.",
    "Very satisfied with my purchase.",
    "Will definitely recommend to others!",
]

# --- URL pool ---
domains = [
    "example.com", "shop.io", "review-hub.net", "feedbacks.org",
    "mystore.co", "buynow.shop", "rateit.app", "quickbuy.biz",
]
paths = ["/product/123", "/item/456", "/review/789", "/order/321", "/listing/654"]

# --- Categorical pools ---
languages = ["English", "Spanish", "French", "German", "Mandarin", "Portuguese", "Arabic"]
lang_weights = [0.40, 0.18, 0.10, 0.08, 0.10, 0.08, 0.06]

countries = [
    "United States", "Mexico", "France", "Germany", "China",
    "Brazil", "Egypt", "United Kingdom", "Canada", "India",
]
country_weights = [0.25, 0.10, 0.08, 0.07, 0.10, 0.08, 0.05, 0.10, 0.08, 0.09]

devices = ["Mobile", "Desktop", "Tablet"]
device_weights = [0.55, 0.35, 0.10]

# --- Build rows ---
text_messages = [random.choice(text_templates) for _ in range(N)]
urls = [
    f"https://www.{random.choice(domains)}{random.choice(paths)}"
    for _ in range(N)
]
lang_col = random.choices(languages, weights=lang_weights, k=N)
country_col = random.choices(countries, weights=country_weights, k=N)
device_col = random.choices(devices, weights=device_weights, k=N)
star_review = np.random.choice([1, 2, 3, 4, 5], size=N, p=[0.05, 0.10, 0.20, 0.35, 0.30])

df = pd.DataFrame({
    "text_message": text_messages,
    "url": urls,
    "language": lang_col,
    "country": country_col,
    "device": device_col,
    "star_review": star_review,
})

print(f"Dataset shape: {df.shape}")
df.head(10)

## 3. Descriptive Statistics

In [ ]:
print("=== Data Types ===")
print(df.dtypes)
print("\n=== Null Counts ===")
print(df.isnull().sum())
print("\n=== Numeric Summary ===")
df.describe()

In [ ]:
# statistics module (built-in Python) on star_review
reviews = df["star_review"].tolist()

print("--- star_review statistics (built-in statistics module) ---")
print(f"  Mean   : {statistics.mean(reviews):.4f}")
print(f"  Median : {statistics.median(reviews):.4f}")
print(f"  Mode   : {statistics.mode(reviews)}")
print(f"  Stdev  : {statistics.stdev(reviews):.4f}")
print(f"  Variance: {statistics.variance(reviews):.4f}")

In [ ]:
# Value counts for categorical columns
for col in ["language", "country", "device", "star_review"]:
    print(f"\n=== {col} ===")
    print(df[col].value_counts())

## 4. Entropy Analysis

Shannon entropy measures the uncertainty / diversity of a categorical distribution.
A higher entropy means values are more evenly distributed; lower entropy means one category dominates.

In [ ]:
def column_entropy(series: pd.Series) -> float:
    """Compute Shannon entropy (bits) for a categorical pandas Series."""
    counts = series.value_counts(normalize=True)
    return float(scipy_entropy(counts, base=2))


categorical_cols = ["language", "country", "device", "star_review", "text_message"]
entropy_results = {col: column_entropy(df[col]) for col in categorical_cols}

entropy_df = pd.DataFrame(
    list(entropy_results.items()), columns=["column", "entropy_bits"]
).sort_values("entropy_bits", ascending=False).reset_index(drop=True)

print(entropy_df.to_string(index=False))

## 5. Visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Synthetic Dataset – Distribution Plots", fontsize=14, fontweight="bold")

# Star review distribution
df["star_review"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0, 0], color="steelblue", edgecolor="white"
)
axes[0, 0].set_title("Star Review Distribution")
axes[0, 0].set_xlabel("Stars")
axes[0, 0].set_ylabel("Count")
axes[0, 0].tick_params(axis="x", rotation=0)

# Device distribution
df["device"].value_counts().plot(
    kind="pie", ax=axes[0, 1], autopct="%1.1f%%", startangle=90
)
axes[0, 1].set_title("Device Distribution")
axes[0, 1].set_ylabel("")

# Language distribution
df["language"].value_counts().plot(
    kind="barh", ax=axes[1, 0], color="coral", edgecolor="white"
)
axes[1, 0].set_title("Language Distribution")
axes[1, 0].set_xlabel("Count")

# Entropy per column
axes[1, 1].barh(
    entropy_df["column"], entropy_df["entropy_bits"], color="mediumseagreen", edgecolor="white"
)
axes[1, 1].set_title("Shannon Entropy per Column (bits)")
axes[1, 1].set_xlabel("Entropy (bits)")

plt.tight_layout()
plt.show()

In [ ]:
# Country distribution
fig, ax = plt.subplots(figsize=(10, 5))
df["country"].value_counts().plot(kind="bar", ax=ax, color="mediumpurple", edgecolor="white")
ax.set_title("Country Distribution")
ax.set_xlabel("Country")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 6. Entropy Breakdown by Category

Compute the entropy of `star_review` conditioned on each categorical column.

In [ ]:
def conditional_entropy(df: pd.DataFrame, condition_col: str, target_col: str) -> float:
    """H(target | condition) = sum_c P(c) * H(target | condition=c)"""
    total = len(df)
    cond_ent = 0.0
    for value, group in df.groupby(condition_col):
        p_c = len(group) / total
        h_given_c = column_entropy(group[target_col])
        cond_ent += p_c * h_given_c
    return cond_ent


base_entropy = column_entropy(df["star_review"])
print(f"Base entropy H(star_review) = {base_entropy:.4f} bits\n")

for col in ["language", "country", "device"]:
    cond_ent = conditional_entropy(df, col, "star_review")
    info_gain = base_entropy - cond_ent
    print(f"H(star_review | {col:10s}) = {cond_ent:.4f} bits  |  Information Gain = {info_gain:.4f} bits")

## 7. Export Dataset

In [ ]:
df.to_csv("./synthetic_dataset.csv", index=False)
print("Dataset saved to synthetic_dataset.csv")
print(df.shape)